# Автоматизация оценки стоимости недвижимости и выявление аномалий

## Бизнес-контекст и задача

В распоряжении находятся исторические данные сервиса Яндекс Недвижимость — архив объявлений о продаже квартир в Санкт-Петербурге и соседних населённых пунктах.

## Цель исследования 

Установить ключевые рыночные параметры, влияющие на ценообразование объектов недвижимости. Результаты работы послужат основой для построения автоматизированной системы оценки, а также алгоритмов триггерного мониторинга аномалий и потенциально мошеннической деятельности.

## Задачи исследования

- **Предобработка данных:**  Выявление природы пропусков, устранение системных ошибок выгрузки, оптимизация типов данных.
- **Инжиниринг признаков:** Расчет удельных стоимостей, временных и локационных метрик.
- **Исследовательский анализ (EDA):** Изучение распределений (площади, цены, высоты потолков), выявление границ «нормального» рынка и отсечение выбросов.
- **Факторный анализ:** Оценка корреляционных и нелинейных зависимостей цены от внутренних характеристик объекта и его географического положения (включая выделение центральной зоны Санкт-Петербурга).


## Подготовка окружения и импорт библиотек

In [ ]:
import os
import warnings
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

import phik
from phik.report import plot_correlation_matrix

warnings.filterwarnings('ignore')
pd.options.display.float_format = '{:.3f}'.format
sns.set_theme(style="whitegrid")

RANDOM_STATE = 12345
URL = 'https://code.s3.yandex.net/datasets/real_estate_data.csv'

## Загрузка данных и первичный экспресс-анализ

In [ ]:
df =pd.read_csv(URL, sep='\t')

In [ ]:
df.info()

In [ ]:
df.head()

## Предобработка данных

### Очистка от явных дубликатов

In [ ]:
initial_rows =df.shape[0]
duplicates_count =df.duplicated().sum()
print(f'Обнаружено явных дубликатов строк: {duplicates_count}')
if duplicates_count> 0:
    df.drop_duplicates().reset_index(drop=True)
    print(f'Явные дубликаты удалены. Удалено строк:{initial_rows - df.shape[0]}')
else:
    print("Явные дубликаты в данных отсутствуют. Идем дальше.")

### Логическое заполнение пропусков

В исходном датасете присутствует значительное количество пропущенных значений.

Для восстановления целостности данных применим логическое заполнение по следующему принципу:

- **`balcony`** $\rightarrow$ `0`: если количество балконов не указано, вероятнее всего, в данной квартире они отсутствуют.
- **`parks_around3000`** $\rightarrow$ `0`: отсутствие данных означает, что в радиусе 3 км нет ни одного парка.
- **`ponds_around3000`** $\rightarrow$ `0`: аналогично паркам, пропуск интерпретируется как отсутствие водоёмов поблизости.
- **`is_apartment`** $\rightarrow$ `False`: если тип недвижимости не заполнен, объект относится к традиционному жилому фонду (не является апартаментами).

*Примечание: такое заполнение позволяет сохранить объём выборки и не искажает реальные рыночные распределения.*


In [ ]:
logical_fills ={
    'balcony':0,
    'parks_around3000': 0,
    'ponds_around3000': 0,
    'is_apartment':False   
}

df.fillna(value=logical_fills, inplace=True)

### Оптимизация типов данных

In [ ]:
types_dict = {
    'balcony':'int',
    'parks_around3000':'int',
    'ponds_around3000':'int',
    'is_apartment': 'bool'
}

df = df.astype(types_dict)

df['first_day_exposition'] = pd.to_datetime(df['first_day_exposition'],format='%Y-%m-%dT%H:%M:%S')

### Очистка названий городов от неявных дубликатов

In [ ]:
df['locality_name'] = df['locality_name'].fillna('')

df['locality_clean'] =(
    df['locality_name']
    .str.replace('ё','е')
    .str.replace(r'\b[а-яё]+\b\s*', '', regex=True)
    .str.strip()
)

### Заполнение пропусков по величине потолков

In [ ]:
ceiling_medians = df.groupby('floors_total')['ceiling_height'].transform('median')
df['ceiling_height'] = df['ceiling_height'].fillna(ceiling_medians)
df['ceiling_height'] = df['ceiling_height'].fillna(df['ceiling_height'].median())

### Проверка результатов предобработки

In [ ]:
print("-" * 50)
print(f"Итоговый размер датасета после очистки: {df.shape[0]} строк.")
print(f"Было уникальных локаций: {df['locality_name'].nunique()}, стало: {df['locality_clean'].nunique()}")

In [ ]:
remaining_na = df.isna().sum()
print("Оставшиеся пропуски по столбцам:")
print(remaining_na[remaining_na > 0])
print("-" * 50)

In [ ]:
print("Типы измененных столбцов:")
print(df[['balcony', 'parks_around3000', 'ponds_around3000', 'is_apartment', 'first_day_exposition']].dtypes)
print("-" * 50)

In [ ]:
data_retention = (len(df) / 23699) * 100
print(f"Сохранено данных после предобработки: {data_retention:.2f}%")

### Удаляем строки, где не указана этажность дома 

In [ ]:
df = df.dropna(subset=['floors_total']).reset_index(drop=True)
df['floors_total'] = df['floors_total'].astype('int')
print(f"Строки без этажности удалены. Текущий размер датасета: {df.shape[0]} строк.")


### Добавление новых признаков (Feature Engineering)

Для подготовки датасета к последующему исследовательскому и факторному анализу расширим таблицу расчетными метриками. 

**Добавляемые параметры:**
1. **`price_per_m`** — удельная стоимость квадратного метра объекта.
2. **Временные маркеры** — день недели (`weekday_exposition`), месяц (`month_exposition`) и год (`year_exposition`) публикации объявления.
3. **`floor_category`** — категориальный признак этажа квартиры (`первый`, `последний` или `другой`).
4. **`center_distance_km`** — расстояние до центра Санкт-Петербурга, переведённое в километры и округлённое до целого числа.


In [ ]:
df['price_per_m'] = df['last_price']/df['total_area'] # расчет стоимости за квадратный метр
df['weekday_exposition'] = df['first_day_exposition'].dt.weekday
df['month_exposition'] = df['first_day_exposition'].dt.month
df['year_exposition'] = df['first_day_exposition'].dt.year

conditions=[
    df['floor'] ==1,
    df['floor'] == df['floors_total']
]
choices = ['первый','последний']

df['floor_category'] = np.select(conditions,choices, default= 'другой')

df['center_distance_km'] = (df['cityCenters_nearest'] /1000).round()

new_columns = ['price_per_m', 'weekday_exposition', 'floor_category', 'center_distance_km']
df[new_columns].head(5)

### Контроль распределения категорий этажей

In [ ]:
print(df['floor_category'].value_counts())
print("-" * 50)
print(df['floor_category'].value_counts(normalize=True))

## Исследовательский анализ данных (EDA)

На данном этапе проведем детальный анализ распределения ключевых параметров объектов недвижимости, определим границы нормальных рыночных значений и выявим скрытые аномальные выбросы.

**Параметры для первичного анализа:**
1. Общая площадь (`total_area`)
2. Жилая площадь (`living_area`)
3. Площадь кухни (`kitchen_area`)
4. Цена объекта (`last_price`)
5. Количество комнат (`rooms`)
6. Высота потолков (`ceiling_height`)

Чтобы избежать дублирования кода, напишем универсальную функцию автоматической визуализации распределений (гистограмма + диаграмма размаха).


In [ ]:
def plot_distribution(df, column, title, bins=50, x_lim=None):
    """
    Функция строит гистограмму распределения и boxplot для заданного признака.
    """
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    
    # 1. Гистограмма распределения с графиком плотности (KDE)
    sns.histplot(data=df, x=column, bins=bins, kde=True, ax=axes[0], color='skyblue')
    axes[0].set_title(f'Гистограмма: {title}')
    axes[0].set_xlabel(title)
    axes[0].set_ylabel('Количество объявлений')
    if x_lim:
        axes[0].set_xlim(x_lim)
        
    # 2. Диаграмма размаха (Boxplot) для наглядного поиска выбросов
    sns.boxplot(data=df, x=column, ax=axes[1], color='lightcoral')
    axes[1].set_title(f'Диаграмма размаха: {title}')
    axes[1].set_xlabel(title)
    if x_lim:
        axes[1].set_xlim(x_lim)
        
    plt.tight_layout()
    plt.show()

In [ ]:
plot_distribution(df, 'total_area', 'Общая площадь (кв. м)', bins=100, x_lim=(0, 300))

На гистограмме слева мы видим классическое распределение с тяжелым правым хвостом. Большинство квартир сосредоточено в диапазоне от 30 до 70 кв.м. Есть выраженный пик в районе 40-45 кв.м (типичные однушки и двушки)

На диаграмме размахов мы видим, что граница нормальных значений заканчивается на  115 кв. м., больше 115 - это выбросы (дворцы, пентхаусы и тд.). Они будут сильно смещать средние значения




In [ ]:
plot_distribution(df, 'living_area', 'Жилая площадь (кв. м)', bins=100, x_lim=(0, 150))

А здесь мы видим мультимодальное распределение, первый пик в районе 17-19 кв.м - это студии


Второй пик в районе 30 кв.м. это однушки еще наблюдаются квартиры с околонулевой площадью - это явные аномалии, либо ошибки ввода данных 

In [ ]:
plot_distribution(df, 'kitchen_area', 'Площадь кухни (кв. м)', bins=50, x_lim=(0, 50))

Снова аномалии около нуля (0-3 кв.м). Это выброс, который нужно будет ограничить снизу (например, отрезать всё, что меньше 4–5 кв. м, либо обрабатывать студии отдельно). Логично поставить фильтр на 4–5 кв. м, чтобы убрать технические ошибки ввода.

По «ящику с усами» справа видно, что граница нормального распределения заканчивается строго на отметке около 19 кв. м. Всё, что выше 20 кв. м — это либо огромные кухни-гостиные в элитных европланировках, либо рестораны/коммерческая недвижимость, попавшая в базу по ошибке

In [ ]:
df['last_price_mln'] = df['last_price'] / 1000000
plot_distribution(df, 'last_price_mln', 'Цена объекта (млн руб.)', bins=100, x_lim=(0, 30))

Верхняя граница типичных рыночных цен) заканчивается ровно на отметке около 12 млн рублей


Основной пик (мода) рынка приходится на диапазон от 3.5 до 5.5 млн рублей. Это абсолютное ядро массового рынка недвижимости в Санкт-Петербурге и Ленинградской области.

Квартир по цене смартфона не бывает — это технические ошибки (например, когда цену указали за квадратный метр или забыли дописать три нуля). Минимальная разумная граница для квартиры в области — около 1 млн рублей.

In [ ]:
plot_distribution(df, 'rooms', 'Количество комнат (шт)', bins=10, x_lim=(0, 10))

Квартир без комнат на рынке не бывает, но в терминологии баз данных недвижимости 0 комнат — это свободная планировка или квартиры-студии

По «ящику с усами» справа видно, что верхняя граница типичного рынка заканчивается ровно на 6 комнатах. Всё, что больше 6 комнат (7, 8, 9, 10+), — это огромные коммуналки, старый фонд или объединённые пентхаусы. Для массовой оценки рынка это явные выбросы.

In [ ]:
plot_distribution(df, 'ceiling_height', 'Высота потолков (м)', bins=50, x_lim=(2, 5))

Основной пик на гистограмме приходится строго на 2.5–2.7 метра. Это абсолютный стандарт советского и современного массового домостроения (хрущёвки, брежневки, стандартные панельные новостройки).

Нижний «ус» заканчивается примерно на 2.2–2.4 м. Потолки ниже 2.4 м в жилых домах — это либо подвальные помещения, либо грубые ошибки ввода. Нам стоит отрезать всё, что ниже 2.4 м.

Верхний «ус» отсекает нормальные значения на уровне 3.0–3.1 м (сталинки и хороший бизнес-класс).

### Очистка данных от выбросов

На основе проведенного графического анализа и построенных диаграмм размаха («ящиков с усами») сформируем единые критерии фильтрации для удаления аномалий:
1. **Высота потолков (`ceiling_height`):** Исправим опечатки ввода (значения от 20 до 40 м поделим на 10), после чего ограничим диапазон значениями от 2.4 до 4.0 м.
2. **Общая площадь (`total_area`):** Ограничим верхний порог на уровне 115 кв. м.
3. **Площадь кухни (`kitchen_area`):** Исключим технический шум около нуля и гигантские кухни, оставив диапазон от 4 до 20 кв. м.
4. **Жилая площадь (`living_area`):** Ограничим диапазон от 10 до 76 кв. м.
5. **Цена объекта (`last_price`):** Оставим массовый сегмент рынка недвижимости в диапазоне от 1.0 до 12.0 млн рублей.
6. **Количество комнат (`rooms`):** Ограничим выборку объектами с числом комнат от 0 до 6 включительно.

Оценим объем отсеченных данных, чтобы убедиться в сохранении репрезентативности выборки.


In [ ]:
rows_before_cleaning = df.shape[0]


df.loc[(df['ceiling_height'] >= 20) & (df['ceiling_height'] <= 40), 'ceiling_height'] /= 10


df_clean = df[
    ((df['ceiling_height'] >= 2.4) & (df['ceiling_height'] <= 4.0)) &
    (df['total_area'] <= 130) &
    ((df['kitchen_area'] >= 4) & (df['kitchen_area'] <= 25) | df['kitchen_area'].isna()) &
    ((df['living_area'] >= 10) & (df['living_area'] <= 85) | df['living_area'].isna()) &
    ((df['last_price'] >= 1_000_000) & (df['last_price'] <= 15_000_000)) &
    (df['rooms'] <= 6)
].reset_index(drop=True)

rows_after_cleaning = df_clean.shape[0]
lost_rows = rows_before_cleaning - rows_after_cleaning
lost_percentage = (lost_rows / rows_before_cleaning) * 100

print(f"Очистка данных успешно завершена!")
print(f"Строк до очистки: {rows_before_cleaning}")
print(f"Строк после очистки: {rows_after_cleaning}")
print(f"Удалено аномальных строк: {lost_rows} ({lost_percentage:.2f}%)")


##  Факторный анализ стоимости недвижимости

На данном этапе мы изучим, какие факторы сильнее всего влияют на ценообразование объектов недвижимости.

**Мы исследуем зависимость полной стоимости (`last_price`) от:**
1. **Количественных параметров:** общая площадь, жилая площадь, площадь кухни, количество комнат.
2. **Категориальных параметров:** тип этажа (`первый`, `последний`, `другой`).
3. **Временных параметров:** день недели, месяц и год публикации объявления.

Для количественных признаков мы построим матрицу корреляции и диаграммы рассеяния, а для категориальных и временных — медианные графики.


In [ ]:
# выбираем столбцы
corr_columns = ['last_price', 'total_area', 'living_area', 'kitchen_area', 'rooms']

corr_matrix = df_clean[corr_columns].corr() #матрица пирсона


plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5) # тепловая карта
plt.title('Матрица корреляции факторов стоимости')
plt.show()

print("Зависимость цены от факторов (коэффициент корреляции):")
print(corr_matrix['last_price'].sort_values(ascending=False))

### Влияние количественных параметров на стоимость

Анализ коэффициентов корреляции Пирсона позволил выявить следующие закономерности:
* **Общая площадь** (`total_area`) имеет наиболее сильное прямое влияние на конечную стоимость объекта ($r = 0.73$).
* **Жилая площадь и площадь кухни** демонстрируют умеренно-сильную связь с ценой ($r = 0.62$ и $r = 0.52$ соответственно). Их вклад значителен, но вторичен по отношению к общему метражу.
* **Количество комнат** оказывает наименьшее линейное влияние на цену ($r = 0.46$). Это объясняется рыночной спецификой: просторные квартиры свободной планировки или современные "евро-форматы" с меньшим числом комнат зачастую превосходят по стоимости тесные многокомнатные объекты старого фонда.


In [ ]:
phik_columns = [
    'last_price', 'total_area', 'living_area', 'kitchen_area', 
    'rooms', 'floor_category', 'weekday_exposition', 'month_exposition', 'year_exposition'
]
interval_features = ['last_price', 'total_area', 'living_area', 'kitchen_area']


phik_matrix = phik.phik_matrix(df_clean[phik_columns], interval_cols=interval_features)


plt.figure(figsize=(10, 8))
sns.heatmap(phik_matrix, annot=True, cmap='YlGnBu', fmt='.2f', linewidths=0.5)
plt.title('Матрица нелинейной корреляции Phik ($\phi_k$)')
plt.show()

### Анализ мультиколлинеарности и нелинейных зависимостей ($\phi_k$)

Применение расширенного коэффициента корреляции Phik ($\phi_k$) позволило оценить совместное влияние числовых, категориальных и временных факторов на стоимость:
* **Категория этажа (`floor_category`):** Выявлена устойчивая нелинейная связь с ценой ($0.20$). Это подтверждает рыночную гипотезу о том, что расположение квартиры (первый/последний этаж против среднего) значимо влияет на её ценность в глазах покупателя.
* **Временные маркеры:** День недели и месяц публикации практически не связаны с ценой объекта ($\leq 0.02$). Однако фактор года (`year_exposition`) показывает корреляцию $0.08$, что обусловлено долгосрочными экономическими трендами и изменением общих цен на рынке недвижимости по годам.
* **Внутренняя мультиколлинеарность:** Наблюдается экстремально высокая связь между общей и жилой площадью ($0.88$). При последующем обучении линейных моделей машинного обучения один из этих признаков (например, `living_area`) целесообразно исключить, чтобы избежать переобучения модели.


In [ ]:
spb_data = df_clean[
    (df_clean['locality_clean'] == 'Санкт-Петербург') & 
    (df_clean['center_distance_km'].notna()) &
    (df_clean['center_distance_km'] <= 20)
]


spb_km_prices = spb_data.groupby('center_distance_km')['price_per_m'].mean().reset_index()


fig, axes = plt.subplots(1, 2, figsize=(18, 7))


sns.kdeplot(
    data=spb_data, 
    x='center_distance_km', 
    y='last_price_mln', 
    cmap='Reds', 
    fill=True, 
    thresh=0.05, 
    ax=axes[0]
)
axes[0].set_title('Плотность распределения рынка (Цена vs Расстояние)')
axes[0].set_xlabel('Расстояние до центра (км)')
axes[0].set_ylabel('Цена объекта (млн руб.)')
axes[0].grid(True, linestyle='--', alpha=0.3)

sns.lineplot(
    data=spb_km_prices, 
    x='center_distance_km', 
    y='price_per_m', 
    marker='o', 
    color='crimson', 
    linewidth=2.5, 
    ax=axes[1]
)
axes[1].set_title('Кривая стоимости: Точка перегиба границ центра')
axes[1].set_xlabel('Расстояние до центра (км)')
axes[1].set_ylabel('Средняя цена кв. м (руб.)')
axes[1].grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()


### Определение границ центральной зоны Санкт-Петербурга

Совместный анализ плотности распределения рынка и динамики стоимости квадратного метра позволил сделать следующие выводы:
* **Географическое ядро рынка:** Основная масса предложений на рынке Санкт-Петербурга сосредоточена в спальных районах на удалении **11–16 км** от центра со средней стоимостью объектов **4.0–5.5 млн рублей**.
* **Выявление границы центра:** На линейном графике зависимости цены от расстояния отчётливо наблюдается точка перегиба на отметке **7 км**. Внутри этого радиуса цена квадратного метра удерживается на максимальном уровне (120–125 тыс. руб.), а после преодоления порога в 7 км происходит резкое падение стоимости, которая к 9–10 км стабилизируется на базовом уровне спальных районов (около 105 тыс. руб.).

<div class="alert alert-block alert-success">
<b>Итоговый инсайт:</b> Радиус центральной зоны Санкт-Петербурга составляет <b>7 км</b>. При построении моделей автоматической оценки недвижимости данный фактор необходимо использовать как ключевой категориальный разделитель рыночных сегментов.
</div>


## Общий вывод по результатам исследования рынка недвижимости

В ходе работы над проектом был проведён комплексный анализ архивных данных сервиса Яндекс Недвижимость по Санкт-Петербургу и Ленинградской области. Код был полностью оптимизирован, устранены системные ошибки выгрузки, и проведено исследование факторов, влияющих на ценообразование.

### 1. Результаты предобработки и подготовки данных
* **Очистка данных:** Удалены скрытые дубликаты в названиях населённых пунктов (унифицировано **306** уникальных локаций вместо исходных 365). Устранены критические опечатки в высоте потолков (значения в диапазоне 20–40 м успешно приведены к реальным 2.0–4.0 м).
* **Фильтрация аномалий:** По результатам графического анализа (диаграмм размаха) была проведена мягкая очистка датасета от рыночного шума. Отсечены квартиры с экстремальными площадями (более 130 кв. м), числом комнат (>6) и ценой (>15 млн руб.). Потери данных составили всего **8.28%**, что полностью сохраняет репрезентативность выборки.
* **Feature Engineering:** Внедрена векторная категоризация этажей (`np.select`), корректно выделившая долю первых (**12.3%**) и последних (**14.1%**) этажей без системных сдвигов для одноэтажных домов.

### 2. Ключевые факторы ценообразования (Факторный анализ)
* **Драйверы стоимости:** Ключевым линейным и нелинейным фактором стоимости является **общая площадь объекта** ($\phi_k = 0.73$). Жилая площадь ($0.64$) и площадь кухни ($0.54$) вносят вторичный вклад. Количество комнат влияет на конечную цену умеренно ($0.40$), уступая фактору общего метража (за счёт популярности современных компактных европланировок).
* **Категориальные зависимости:** Подтверждена гипотеза нелинейного влияния этажа квартиры на её стоимость ($\phi_k = 0.20$). Объекты на первом и последнем этажах торгуются с заметным дисконтом относительно средних этажей.
* **Временные тренды:** Сезонность внутри года (день недели или месяц публикации) не оказывает значимого влияния на цену недвижимости ($\phi_k \leq 0.02$). При этом макроэкономические изменения по годам имеют выраженный характер ($\phi_k = 0.08$).

### 3. Географические границы центральной зоны
* На основе сопоставления плотности рынка и кривой стоимости квадратного метра была математически определена **граница исторического центра Санкт-Петербурга — она ограничена радиусом в 7 км** от центральной точки.
* Внутри 7-километровой зоны цена квадратного метра удерживается на пиковом уровне (**120 000 — 125 000 руб.**), после чего происходит обвальное падение. К 9–10 км стоимость стабилизируется на уровне массовых спальных районов (**~105 000 руб.**). Основное ядро предложений (концентрация спального фонда) сосредоточено на удалении 11–16 км от центра.

### 🎯 Бизнес-рекомендации для автоматизированной системы оценки:
1. Использовать **общую площадь** как базовый непрерывный признак для расчёта стоимости.
2. Внедрить в алгоритм жёсткий бинарный флаг **«Внутри центрального радиуса (до 7 км)»**, так как ценообразование внутри и снаружи этой границы подчиняется разным рыночным законам.
3. Обязательно учитывать понижающие коэффициенты для категорий **«первый»** и **«последний»** этаж.


<div class="alert alert-success">
<b>Комментарий ревьюера: ✅</b>

Здорово, что оцениваешь доли пропущенных значений 👍, ведь так гораздо быстрее разобраться где больше всего пропусков.

</div>

In [ ]:
propusk['isna_percentage'].sort_values(ascending=False).plot(kind='bar',grid=True).set_title(
'Количество пропусков по полям, в процентах')
plt.xticks(rotation=90)
plt.show()


Возьмем расшифровку полей из задания  и дополнительную информацию о данных:
По каждой квартире на продажу доступны два вида данных. Первые вписаны пользователем, вторые — получены автоматически на основе картографических данных. Например, расстояние до центра, аэропорта, ближайшего парка и водоёма.


is_apartment — апартаменты (булев тип) - пропуски- нулевые значения (заменяем нулями, меняем тип)

parks_nearest — расстояние до ближайшего парка (м) 
Данные из автоматизированной системы, значит ближайший парк находится 
слишком далеко заменим пропущенные значения на (-1), т.к. при заменене на 0 может исказиться картина


ponds_nearest — расстояние до ближайшего водоёма (м) - ближайший водоем находится 
слишком далеко заменим пропущенные значения на (-1), т.к. при заменене на 0 может исказиться картина

balcony — число балконов. Переводим в инт, заменяем на 0, т.к. это значит, что балконов нет

ceiling_height — высота потолков (м). Заменяем на -1 или на медианы по группам  (число комнат, площадь, жилая площадь). Также можно исключить при анализе по данному критерию. по всей видимости, заполняется пользователем, но пользователь может и не знать высоту.

airports_nearest — расстояние до ближайшего аэропорта в метрах (м) заменяем на -1, т.к. вероятно до ближайшего аэропорта слишком далеко


cityCenters_nearest — расстояние до центра города (м). Видимо, данные с автоматической выгрузки, значит ближайший  центр города находится слишком далеко ? Заменяем на -1



ponds_around3000 — число водоёмов в радиусе 3 км - заменяем 0

parks_around3000 — число парков в радиусе 3 км - заменяем 0


days_exposition — сколько дней было размещено объявление (от публикации до снятия) заменяем на -1

kitchen_area — площадь кухни в квадратных метрах (м²)- либо дропаем, либо заменяем на -1

living_area — жилая площадь в квадратных метрах (м²) - либо дропаем, либо заменяем на -1
floors_total — всего этажей в доме - дропаем

Проведем проверку на явные дубликаты

In [ ]:
display('явных дубликатов', df.duplicated().sum()) #посчет явных дубликатов

Рассмотрим типы данных

In [ ]:
display(df.dtypes) 

is_apartment -неверный формат, должен быть bool,  

floors_total  - неверный формат, должен быть int64. 

first_day_exposition - неверный формат, либо int64 либо date_time(days).

balcony - неверный формат должен быть int 64 (не бывает 2.5 балкона), 

Неверный формат, должен быть  int 64 для:


parks_around3000        float64


parks_nearest           float64


ponds_around3000        float64


ponds_nearest           float64


days_exposition         float64
